# LightGCN for Recommendation — End-to-End Pipeline (with SGL Preprocess & Evaluation)

This notebook implements **LightGCN** from the SIGIR 2020 paper:
> **LightGCN: Simplifying and Powering Graph Convolution Network for Recommendation**  
> Xiangnan He, Kuan Deng, Xiang Wang, Yan Li, YongDong Zhang, and Meng Wang.  
> *SIGIR 2020* | [Official Repository (gusye1234/LightGCN-PyTorch)](https://github.com/gusye1234/LightGCN-PyTorch)

while keeping the **exact same data preprocessing, 7:1:2 (`train:val:test`) split from `yelp2018.train`, BPR sampling, and evaluation pipeline** as our SGL implementation for a direct, controlled comparison.

### Pipeline Structure
1. **Environment & Reproducibility Setup**: Imports, device auto-detection (CUDA / MPS / CPU), and random seeds.
2. **Data Preprocessing (`yelp2018.train` 7:1:2 Split, Graph Laplacian & BPR Sampler)**:
   - Automatic downloader for **`yelp2018.train` only** from the `SGL-Torch` repository (plus a synthetic fallback).
   - Per-user **7 : 1 : 2 (`train : val : test`) interaction split** (70% training, 10% validation, 20% testing) from `yelp2018.train`.
   - Symmetrically normalized bipartite adjacency matrix $\tilde{\mathbf{A}} = \mathbf{D}^{-1/2} \mathbf{A} \mathbf{D}^{-1/2}$ built strictly from the 70% training split.
   - Mini-batch BPR pairwise sampler $(u, i, j)$.
3. **Model Architecture (`LightGCN` from `LightGCN-PyTorch`)**:
   - Follow `gusye1234/LightGCN-PyTorch/code/model.py`:
     - Light Graph Convolution (`computer()`): $\mathbf{E}^{(k+1)} = \tilde{\mathbf{A}} \mathbf{E}^{(k)}$ without feature transformation matrices or non-linearities.
     - Layer combination: $\mathbf{E} = \frac{1}{K+1}\sum_{k=0}^{K} \mathbf{E}^{(k)}$ (`torch.mean(embs, dim=1)`).
     - Optional adjacency dropout (`__dropout` / `__dropout_x`) during training.
     - BPR Loss + $L_2$ regularization on ego embeddings (`bpr_loss()`), plus `predict()` interface for the shared `Evaluator`.
4. **Evaluation (`Evaluator`)**:
   - All-item ranking protocol for `Precision@K`, `Recall@K`, and `NDCG@K` on both the **Validation (10%)** and **Test (20%)** splits, plus the 10-group **Long-Tail Item Group Recall** analysis (Section 4.3.1 of the SGL paper).
5. **Training Loop & Visualization**:
   - End-to-end execution using the 10% validation split for early stopping and checkpointing, followed by final evaluation on the 20% test split.

In [1]:
# ==============================================================================
# 1. Environment Setup & Imports (Colab Ready)
# ==============================================================================
import math
import os
import random
import urllib.request
from collections import defaultdict
from dataclasses import dataclass, field
from time import time

import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F


def set_seed(seed: int = 2021) -> None:
    """Fixes random seeds across Python, NumPy, and PyTorch (CPU & CUDA) for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True


DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__} | Running on Device: {DEVICE}")

## 2. Data Preprocessing (`SGLDataset` & `PairwiseSampler`)

This section uses the **exact same data preprocessing pipeline** as our SGL notebook: loading **`yelp2018.train` only**, splitting each user's interactions into **train : val : test = 7 : 1 : 2** (70% / 10% / 20%), building the symmetrically normalized bipartite adjacency matrix $\tilde{\mathbf{A}} = \mathbf{D}^{-1/2}\mathbf{A}\mathbf{D}^{-1/2}$ on the 70% training split, and sampling BPR triplets $(u, i, j)$.

In [6]:
# ==============================================================================
# 2. Data Preprocessing: Single-File (yelp2018.train) 7:1:2 Split & BPR Sampler
# ==============================================================================

SGL_DATASET_BASE_URL = "https://raw.githubusercontent.com/wujcan/SGL-Torch/master/dataset"


def download_sgl_dataset(data_dir: str, dataset_name: str = "yelp2018") -> str:
    """Downloads ONLY `<dataset_name>.train` from the official SGL-Torch GitHub repo."""
    os.makedirs(data_dir, exist_ok=True)
    train_path = os.path.join(data_dir, f"{dataset_name}.train")
    if not os.path.exists(train_path):
        url = f"{SGL_DATASET_BASE_URL}/{dataset_name}/{dataset_name}.train"
        print(f"[Dataset] Downloading {dataset_name}.train from {url} ...")
        urllib.request.urlretrieve(url, train_path)
    return train_path


def generate_synthetic_dataset(
    data_dir: str,
    num_users: int = 300,
    num_items: int = 500,
    seed: int = 2021,
) -> str:
    """Generates a power-law synthetic `.train` file (each line: `user_id item_id`)."""
    os.makedirs(data_dir, exist_ok=True)
    rng = np.random.default_rng(seed)

    # Power-law item popularity distribution p(i) ~ (i + 10)^(-0.8)
    ranks = np.arange(1, num_items + 1, dtype=np.float64)
    item_probs = (ranks + 10.0) ** (-0.8)
    item_probs /= item_probs.sum()

    train_path = os.path.join(data_dir, "synthetic.train")
    with open(train_path, "w", encoding="utf-8") as f_train:
        for u in range(num_users):
            n_inter = int(rng.integers(15, 35))
            items = rng.choice(num_items, size=n_inter, replace=False, p=item_probs)
            for i in items:
                f_train.write(f"{u} {int(i)}\n")
    return train_path


def sp_mat_to_sp_tensor(sp_mat: sp.spmatrix, device: torch.device = DEVICE) -> torch.Tensor:
    """
    Converts a SciPy sparse matrix into a coalesced PyTorch sparse_coo_tensor.
    Equivalent to `_convert_sp_mat_to_sp_tensor` in SGL-Torch / LightGCN-PyTorch.
    """
    coo = sp_mat.tocoo().astype(np.float32)
    indices = torch.from_numpy(np.vstack((coo.row, coo.col)).astype(np.int64))
    values = torch.from_numpy(coo.data)
    shape = torch.Size(coo.shape)
    return torch.sparse_coo_tensor(indices, values, shape, device=device).coalesce()


class SGLDataset:
    """
    Loads `<dataset_name>.train` ONLY (e.g. `yelp2018.train`), splits each user's
    interactions into train : val : test with ratio 7 : 1 : 2 (70% / 10% / 20%),
    and constructs the intact normalized bipartite adjacency matrix using ONLY the
    70% training split for LightGCN.
    """

    def __init__(
        self,
        data_dir: str = "./dataset/yelp2018",
        dataset_name: str = "yelp2018",
        split_ratios: tuple[float, float, float] = (0.7, 0.1, 0.2),
        seed: int = 2021,
        device: torch.device = DEVICE,
    ) -> None:
        self.data_dir = data_dir
        self.dataset_name = dataset_name
        self.split_ratios = split_ratios
        self.seed = seed
        self.device = device

        # 1. Resolve `<dataset_name>.train` path (ignores `.test` completely)
        if dataset_name == "synthetic":
            train_path = os.path.join(data_dir, "synthetic.train")
            if not os.path.exists(train_path):
                train_path = generate_synthetic_dataset(data_dir, seed=seed)
        else:
            train_path = os.path.join(data_dir, f"{dataset_name}.train")
            if not os.path.exists(train_path):
                train_path = download_sgl_dataset(data_dir, dataset_name)

        # 2. Load all raw interactions from `<dataset_name>.train` and split 7:1:2 per user
        (
            self.num_users,
            self.num_items,
            self.total_interactions,
            self.train_users,
            self.train_items,
            self.train_user_dict,
            self.val_user_dict,
            self.test_user_dict,
            self.train_val_user_dict,
            self.train_user_set,
        ) = self._load_and_split_train_file(train_path, split_ratios=split_ratios, seed=seed)

        self.num_nodes = self.num_users + self.num_items
        self.num_ratings = len(self.train_users)
        self.num_val_ratings = sum(len(v) for v in self.val_user_dict.values())
        self.num_test_ratings = sum(len(v) for v in self.test_user_dict.values())

        # 3. Pre-construct the intact normalized adjacency matrix G using ONLY the 70% train split
        norm_adj_sp = self.create_adj_mat(self.train_users, self.train_items)
        self.norm_adj = sp_mat_to_sp_tensor(norm_adj_sp, device=self.device)

    @property
    def n_users(self) -> int:
        return self.num_users

    @property
    def m_items(self) -> int:
        return self.num_items

    def getSparseGraph(self) -> torch.Tensor:
        """Compatibility method matching `BasicDataset.getSparseGraph()` in LightGCN-PyTorch."""
        return self.norm_adj

    @staticmethod
    def _load_and_split_train_file(
        train_path: str,
        split_ratios: tuple[float, float, float] = (0.7, 0.1, 0.2),
        seed: int = 2021,
    ):
        """
        Reads user-item interactions from `<dataset_name>.train` ONLY and splits each
        user's interactions into train : val : test according to `split_ratios` (7:1:2).
        """
        raw_user_items = defaultdict(list)
        max_u, max_i = -1, -1

        with open(train_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split(",")
                u = int(parts[0])
                max_u = max(max_u, u)
                for item_str in parts[1:]:
                    i = int(item_str)
                    raw_user_items[u].append(i)
                    max_i = max(max_i, i)

        num_users = max_u + 1
        num_items = max_i + 1

        # Normalize ratios to sum to 1.0 (e.g., (7, 1, 2) or (0.7, 0.1, 0.2) -> 0.7, 0.1, 0.2)
        r_sum = float(sum(split_ratios))
        train_ratio = split_ratios[0] / r_sum
        val_ratio = split_ratios[1] / r_sum

        rng = np.random.default_rng(seed)

        train_users, train_items = [], []
        train_user_dict = {}
        val_user_dict = {}
        test_user_dict = {}
        train_val_user_dict = {}
        train_user_set = defaultdict(set)
        total_interactions = 0

        for u in range(num_users):
            items_list = raw_user_items.get(u, [])
            if not items_list:
                continue
            # Deduplicate while preserving initial order before shuffling
            items = np.array(list(dict.fromkeys(items_list)), dtype=np.int32)
            rng.shuffle(items)
            n = len(items)
            total_interactions += n

            if n >= 3:
                n_train = max(1, int(round(train_ratio * n)))
                n_val = max(1, int(round(val_ratio * n)))
                if n_train + n_val >= n:
                    n_train = max(1, n - 2)
                    n_val = 1
                u_train = items[:n_train]
                u_val = items[n_train : n_train + n_val]
                u_test = items[n_train + n_val :]
            elif n == 2:
                u_train = items[:1]
                u_val = np.array([], dtype=np.int32)
                u_test = items[1:]
            else:
                u_train = items
                u_val = np.array([], dtype=np.int32)
                u_test = np.array([], dtype=np.int32)

            train_user_dict[u] = u_train
            if len(u_val) > 0:
                val_user_dict[u] = u_val
            if len(u_test) > 0:
                test_user_dict[u] = u_test

            # Combined train + val items per user for masking during final Test evaluation
            train_val_user_dict[u] = (
                np.concatenate([u_train, u_val]) if len(u_val) > 0 else u_train
            )

            train_user_set[u] = set(u_train.tolist())
            train_users.extend([u] * len(u_train))
            train_items.extend(u_train.tolist())

        return (
            num_users,
            num_items,
            total_interactions,
            np.asarray(train_users, dtype=np.int32),
            np.asarray(train_items, dtype=np.int32),
            train_user_dict,
            val_user_dict,
            test_user_dict,
            train_val_user_dict,
            train_user_set,
        )

    def create_adj_mat(self, users: np.ndarray, items: np.ndarray) -> sp.csr_matrix:
        """
        Constructs the symmetrically normalized bipartite adjacency matrix:
            A_tilde = D^{-1/2} * A * D^{-1/2}
        """
        user_np = np.array(users)
        item_np = np.array(items) + self.num_users
        ratings = np.ones_like(user_np, dtype=np.float32)

        row = np.concatenate([user_np, item_np])
        col = np.concatenate([item_np, user_np])
        data = np.concatenate([ratings, ratings])
        adj_mat = sp.csr_matrix((data, (row, col)), shape=(self.num_nodes, self.num_nodes))

        rowsum = np.array(adj_mat.sum(axis=1)).squeeze()
        with np.errstate(divide="ignore"):
            d_inv = np.power(rowsum, -0.5).flatten()
        d_inv[np.isinf(d_inv)] = 0.0
        d_mat_inv = sp.diags(d_inv)
        norm_adj_tmp = d_mat_inv.dot(adj_mat)
        adj_matrix = norm_adj_tmp.dot(d_mat_inv)
        return adj_matrix.tocsr()


class PairwiseSampler:
    """
    Mini-batch sampler for Bayesian Personalized Ranking (BPR) loss.
    For each positive training interaction (u, i) in the 70% train split,
    uniformly samples a negative item j not in u's training history.
    """

    def __init__(
        self, dataset: SGLDataset, batch_size: int = 2048, shuffle: bool = True
    ) -> None:
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.num_users = dataset.num_users
        self.num_items = dataset.num_items
        self.train_users = dataset.train_users
        self.train_items = dataset.train_items
        self.train_user_set = dataset.train_user_set
        self.num_ratings = len(self.train_users)

    def __len__(self) -> int:
        return math.ceil(self.num_ratings / self.batch_size)

    def __iter__(self):
        indices = np.arange(self.num_ratings)
        if self.shuffle:
            np.random.shuffle(indices)

        users_shuffled = self.train_users[indices]
        pos_items_shuffled = self.train_items[indices]

        # Sample one negative item per positive training interaction
        neg_items = np.empty(self.num_ratings, dtype=np.int32)
        for idx, u in enumerate(users_shuffled):
            pos_set = self.train_user_set[int(u)]
            j = random.randint(0, self.num_items - 1)
            while j in pos_set:
                j = random.randint(0, self.num_items - 1)
            neg_items[idx] = j

        for start in range(0, self.num_ratings, self.batch_size):
            end = min(start + self.batch_size, self.num_ratings)
            yield (
                users_shuffled[start:end],
                pos_items_shuffled[start:end],
                neg_items[start:end],
            )


## 3. LightGCN Model Architecture (`gusye1234/LightGCN-PyTorch`)

Below is the PyTorch implementation of **LightGCN** from [`gusye1234/LightGCN-PyTorch/code/model.py`](https://github.com/gusye1234/LightGCN-PyTorch/blob/master/code/model.py):
1. **Weight Initialization (`__init_weight`)**:
   - Creates `self.embedding_user` and `self.embedding_item`.
   - Initializes weights with $\mathcal{N}(0, 0.1^2)$ (`nn.init.normal_(..., std=0.1)`) by default, or `xavier_uniform_` if specified.
2. **Sparse Edge Dropout (`__dropout_x` & `__dropout`)**:
   - When `dropout=1` during training, randomly retains `keep_prob` fraction of non-zero entries in `self.Graph` and rescales kept values by `1 / keep_prob`.
3. **Light Graph Convolution (`computer()`)**:
   - Concatenates user and item 0-th layer embeddings $E^{(0)} = [E_U^{(0)}; E_I^{(0)}]$.
   - Propagates $E^{(k+1)} = \tilde{A} E^{(k)}$ for `n_layers` layers using `torch.sparse.mm`.
   - Combines all layers via uniform average (`torch.mean(torch.stack(embs, dim=1), dim=1)`).
4. **BPR Loss & $L_2$ Regularization (`getEmbedding` & `bpr_loss`)**:
   - Computes pairwise BPR loss using `torch.nn.functional.softplus(neg_scores - pos_scores)`.
   - Computes $L_2$ regularization on the 0-th layer ego embeddings (`userEmb0`, `posEmb0`, `negEmb0`).

In [3]:
# ==============================================================================
# 3. LightGCN Model Implementation (from gusye1234/LightGCN-PyTorch/code/model.py)
# ==============================================================================

class BasicModel(nn.Module):
    def __init__(self):
        super().__init__()

    def getUsersRating(self, users: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError


class LightGCN(BasicModel):
    """
    PyTorch implementation of LightGCN (He et al., SIGIR 2020) from:
    https://github.com/gusye1234/LightGCN-PyTorch/blob/master/code/model.py

    Also includes `predict(users)` and cached eval embeddings so it works directly
    with the shared `Evaluator` from `SGL_Colab.ipynb`.
    """

    def __init__(self, config: dict, dataset: SGLDataset):
        super().__init__()
        self.config = config
        self.dataset = dataset
        self._user_embeddings_final = None
        self._item_embeddings_final = None
        self.__init_weight()

    def __init_weight(self) -> None:
        self.num_users = self.dataset.n_users
        self.num_items = self.dataset.m_items
        self.latent_dim = self.config["latent_dim_rec"]
        self.n_layers = self.config["lightGCN_n_layers"]
        self.keep_prob = self.config.get("keep_prob", 1.0)
        self.A_split = self.config.get("A_split", False)

        # 0-th layer ego ID embeddings
        self.embedding_user = torch.nn.Embedding(
            num_embeddings=self.num_users, embedding_dim=self.latent_dim
        )
        self.embedding_item = torch.nn.Embedding(
            num_embeddings=self.num_items, embedding_dim=self.latent_dim
        )

        if self.config.get("pretrain", 0) == 0:
            init_method = self.config.get("param_init", "normal")
            if init_method == "xavier_uniform":
                nn.init.xavier_uniform_(self.embedding_user.weight, gain=1)
                nn.init.xavier_uniform_(self.embedding_item.weight, gain=1)
                print("[LightGCN] Initialized embeddings with Xavier Uniform.")
            else:
                # Default in gusye1234/LightGCN-PyTorch: Normal N(0, 0.1^2)
                nn.init.normal_(self.embedding_user.weight, std=0.1)
                nn.init.normal_(self.embedding_item.weight, std=0.1)
                print("[LightGCN] Initialized embeddings with Normal(0, 0.1).")
        else:
            self.embedding_user.weight.data.copy_(torch.from_numpy(self.config["user_emb"]))
            self.embedding_item.weight.data.copy_(torch.from_numpy(self.config["item_emb"]))
            print("[LightGCN] Loaded pretrained embeddings.")

        self.f = nn.Sigmoid()
        self.Graph = self.dataset.getSparseGraph()
        print(f"[LightGCN] Ready (n_layers={self.n_layers}, dropout={self.config.get('dropout', 0)}).")

    def __dropout_x(self, x: torch.Tensor, keep_prob: float) -> torch.Tensor:
        """
        Drops edges from sparse tensor `x` with keep probability `keep_prob`
        and rescales surviving edge weights by `1 / keep_prob`.
        """
        x = x.coalesce()
        size = x.size()
        index = x.indices().t()
        values = x.values()
        random_index = torch.rand(len(values), device=values.device) + keep_prob
        random_index = random_index.int().bool()
        index = index[random_index]
        values = values[random_index] / keep_prob
        g = torch.sparse_coo_tensor(index.t(), values, size, device=values.device).coalesce()
        return g

    def __dropout(self, keep_prob: float) -> torch.Tensor | list[torch.Tensor]:
        if self.A_split:
            graph = []
            for g in self.Graph:
                graph.append(self.__dropout_x(g, keep_prob))
        else:
            graph = self.__dropout_x(self.Graph, keep_prob)
        return graph

    def computer(self) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Propagates user and item embeddings across `self.n_layers` LightGCN layers:
            E^{(k+1)} = D^{-1/2} A D^{-1/2} E^{(k)}
            E = (1 / (K + 1)) * sum_{k=0}^{K} E^{(k)}
        """
        users_emb = self.embedding_user.weight
        items_emb = self.embedding_item.weight
        all_emb = torch.cat([users_emb, items_emb])
        embs = [all_emb]

        if self.config.get("dropout", 0) and self.training:
            g_droped = self.__dropout(self.keep_prob)
        else:
            g_droped = self.Graph

        for _ in range(self.n_layers):
            if self.A_split:
                temp_emb = []
                for f in range(len(g_droped)):
                    temp_emb.append(torch.sparse.mm(g_droped[f], all_emb))
                side_emb = torch.cat(temp_emb, dim=0)
                all_emb = side_emb
            else:
                all_emb = torch.sparse.mm(g_droped, all_emb)
            embs.append(all_emb)

        embs = torch.stack(embs, dim=1)
        light_out = torch.mean(embs, dim=1)
        users, items = torch.split(light_out, [self.num_users, self.num_items])
        return users, items

    def getUsersRating(self, users: torch.Tensor) -> torch.Tensor:
        """Computes user-item rating scores across all items using `computer()`."""
        all_users, all_items = self.computer()
        users_emb = all_users[users.long()]
        items_emb = all_items
        rating = self.f(torch.matmul(users_emb, items_emb.t()))
        return rating

    def getEmbedding(
        self, users: torch.Tensor, pos_items: torch.Tensor, neg_items: torch.Tensor
    ) -> tuple[torch.Tensor, ...]:
        """
        Returns both final multi-layer propagated embeddings and 0-th layer ego embeddings
        for the given batch of (users, pos_items, neg_items).
        """
        all_users, all_items = self.computer()
        users_emb = all_users[users]
        pos_emb = all_items[pos_items]
        neg_emb = all_items[neg_items]

        users_emb_ego = self.embedding_user(users)
        pos_emb_ego = self.embedding_item(pos_items)
        neg_emb_ego = self.embedding_item(neg_items)
        return users_emb, pos_emb, neg_emb, users_emb_ego, pos_emb_ego, neg_emb_ego

    def bpr_loss(
        self, users: torch.Tensor, pos: torch.Tensor, neg: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Computes the pairwise Bayesian Personalized Ranking (BPR) loss and L2 regularization
        loss exactly as in `gusye1234/LightGCN-PyTorch/code/model.py`:
            loss     = mean(softplus(neg_scores - pos_scores))
            reg_loss = (1/2) * (||u^{(0)}||_2^2 + ||i^{(0)}||_2^2 + ||j^{(0)}||_2^2) / |B|
        """
        users_emb, pos_emb, neg_emb, userEmb0, posEmb0, negEmb0 = self.getEmbedding(
            users.long(), pos.long(), neg.long()
        )
        reg_loss = (
            0.5
            * (userEmb0.norm(2).pow(2) + posEmb0.norm(2).pow(2) + negEmb0.norm(2).pow(2))
            / float(len(users))
        )

        pos_scores = torch.mul(users_emb, pos_emb)
        pos_scores = torch.sum(pos_scores, dim=1)
        neg_scores = torch.mul(users_emb, neg_emb)
        neg_scores = torch.sum(neg_scores, dim=1)

        loss = torch.mean(torch.nn.functional.softplus(neg_scores - pos_scores))
        return loss, reg_loss

    def forward(self, users: torch.Tensor, items: torch.Tensor) -> torch.Tensor:
        all_users, all_items = self.computer()
        users_emb = all_users[users]
        items_emb = all_items[items]
        inner_pro = torch.mul(users_emb, items_emb)
        gamma = torch.sum(inner_pro, dim=1)
        return gamma

    def eval(self):
        """Precomputes and caches `computer()` embeddings when switching to eval mode."""
        super().eval()
        with torch.no_grad():
            self._user_embeddings_final, self._item_embeddings_final = self.computer()
        return self

    @torch.no_grad()
    def predict(self, users: np.ndarray | list[int] | torch.Tensor) -> np.ndarray:
        """
        Adapter method for the shared `Evaluator`: computes full-ranking scores for `users`
        across all items and returns a NumPy array.
        """
        if not isinstance(users, torch.Tensor):
            device = self.embedding_user.weight.device
            users = torch.from_numpy(np.asarray(users)).long().to(device)
        if self._user_embeddings_final is None or self._item_embeddings_final is None:
            self._user_embeddings_final, self._item_embeddings_final = self.computer()
        users_emb = self._user_embeddings_final[users.long()]
        ratings = torch.matmul(users_emb, self._item_embeddings_final.t())
        return ratings.cpu().detach().numpy()

## 4. All-Ranking Evaluation Protocol & Long-Tail Group Analysis (`Evaluator` — Shared Pipeline)

Uses the exact same `Evaluator` and `compute_user_metrics` implementation as `SGL_Colab.ipynb`:
- Masks out training items (`-np.inf`) and evaluates `Precision@20`, `Recall@20`, and `NDCG@20` over all items.
- Computes the 10-group Long-Tail Item Popularity `Recall@20` decomposition.

In [4]:
# ==============================================================================
# 4. Top-K All-Ranking Evaluator & Long-Tail Group Decomposition (Shared)
# ==============================================================================

def compute_user_metrics(
    ranked_items: np.ndarray,
    ground_truth: set[int],
    top_k_list: list[int],
) -> dict[str, np.ndarray]:
    """
    Computes Precision@K, Recall@K, and NDCG@K for a single user u.
    """
    max_k = len(ranked_items)
    truth_len = len(ground_truth)

    hits = np.array(
        [1.0 if item in ground_truth else 0.0 for item in ranked_items], dtype=np.float64
    )
    cum_hits = np.cumsum(hits)
    ranks = np.arange(1, max_k + 1, dtype=np.float64)

    precisions = cum_hits / ranks
    recalls = cum_hits / truth_len if truth_len > 0 else np.zeros_like(cum_hits)

    discounts = 1.0 / np.log2(np.arange(2, max_k + 2, dtype=np.float64))
    dcg = np.cumsum(hits * discounts)

    ideal_hits = np.zeros(max_k, dtype=np.float64)
    ideal_hits[: min(max_k, truth_len)] = 1.0
    idcg = np.cumsum(ideal_hits * discounts)
    ndcgs = np.where(idcg > 0, dcg / idcg, 0.0)

    k_indices = [k - 1 for k in top_k_list]
    return {
        "Precision": precisions[k_indices],
        "Recall": recalls[k_indices],
        "NDCG": ndcgs[k_indices],
    }


class Evaluator:
    """All-ranking Top-K Evaluator (shared with SGL_Colab.ipynb)."""

    def __init__(
        self,
        user_train_dict: dict[int, np.ndarray],
        user_test_dict: dict[int, np.ndarray],
        metrics: list[str] | None = None,
        top_k: int | list[int] = 20,
        batch_size: int = 128,
    ):
        if metrics is None:
            metrics = ["Precision", "Recall", "NDCG"]
        elif isinstance(metrics, str):
            metrics = [metrics]

        self.user_pos_train = user_train_dict if user_train_dict is not None else {}
        self.user_pos_test = user_test_dict
        self.metrics = metrics
        self.batch_size = batch_size
        self.top_show = [top_k] if isinstance(top_k, int) else sorted(list(top_k))
        self.max_top = max(self.top_show)

    def metrics_info(self) -> str:
        headers = []
        for k in self.top_show:
            for m in self.metrics:
                headers.append(f"{m}@{k}".ljust(12))
        return "\t".join(headers)

    def evaluate(self, model, test_users: list[int] | None = None) -> tuple[np.ndarray, str]:
        model.eval()
        if test_users is None:
            test_users = [u for u, items in self.user_pos_test.items() if len(items) > 0]

        all_user_results = []

        for start in range(0, len(test_users), self.batch_size):
            batch_users = test_users[start : start + self.batch_size]
            ranking_score = model.predict(batch_users)
            ranking_score = np.array(ranking_score, dtype=np.float32, copy=True)

            # Mask observed training items with -inf so they are never recommended
            for idx, u in enumerate(batch_users):
                if u in self.user_pos_train and len(self.user_pos_train[u]) > 0:
                    ranking_score[idx, self.user_pos_train[u]] = -np.inf

            top_k_part = np.argpartition(-ranking_score, self.max_top - 1, axis=1)[:, : self.max_top]
            row_indices = np.arange(len(batch_users))[:, None]
            top_k_scores = ranking_score[row_indices, top_k_part]
            top_k_order = np.argsort(-top_k_scores, axis=1)
            top_k_items = top_k_part[row_indices, top_k_order]

            for idx, u in enumerate(batch_users):
                truth_set = set(self.user_pos_test[u].tolist())
                user_metric_dict = compute_user_metrics(
                    ranked_items=top_k_items[idx],
                    ground_truth=truth_set,
                    top_k_list=self.top_show,
                )
                user_vec = []
                for k_idx in range(len(self.top_show)):
                    for m in self.metrics:
                        user_vec.append(user_metric_dict[m][k_idx])
                all_user_results.append(user_vec)

        all_user_results = np.asarray(all_user_results, dtype=np.float64)
        final_result = np.mean(all_user_results, axis=0)
        buf = "\t".join([("%.8f" % x).ljust(12) for x in final_result])
        return final_result, buf

    def evaluate_long_tail_groups(
        self,
        model,
        train_items: np.ndarray,
        num_items: int,
        num_groups: int = 10,
        k: int = 20,
    ) -> np.ndarray:
        """
        Decomposes Recall@K across 10 item popularity groups (Section 4.3.1, Figure 4).
        Group 1 = lowest-degree (long-tail) items; Group 10 = highest-degree (head) items.
        """
        model.eval()
        item_degrees = np.bincount(train_items, minlength=num_items)
        sorted_items = np.argsort(item_degrees)
        sorted_degrees = item_degrees[sorted_items]
        cum_degrees = np.cumsum(sorted_degrees)
        total_interactions = cum_degrees[-1]

        item_to_group = np.zeros(num_items, dtype=np.int32)
        thresholds = np.linspace(0, total_interactions, num_groups + 1)[1:]
        group_ids = np.clip(np.searchsorted(thresholds, cum_degrees, side="left"), 0, num_groups - 1)
        item_to_group[sorted_items] = group_ids

        test_users = [u for u, items in self.user_pos_test.items() if len(items) > 0]
        group_recalls = np.zeros(num_groups, dtype=np.float64)

        for start in range(0, len(test_users), self.batch_size):
            batch_users = test_users[start : start + self.batch_size]
            ranking_score = np.array(model.predict(batch_users), dtype=np.float32, copy=True)

            for idx, u in enumerate(batch_users):
                if u in self.user_pos_train and len(self.user_pos_train[u]) > 0:
                    ranking_score[idx, self.user_pos_train[u]] = -np.inf

            top_k_part = np.argpartition(-ranking_score, k - 1, axis=1)[:, :k]

            for idx, u in enumerate(batch_users):
                truth_set = set(self.user_pos_test[u].tolist())
                truth_len = len(truth_set)
                for rec_item in top_k_part[idx]:
                    if rec_item in truth_set:
                        g = item_to_group[int(rec_item)]
                        group_recalls[g] += 1.0 / truth_len

        group_recalls /= max(1, len(test_users))
        return group_recalls

## 5. Configuration, Training Pipeline & Visualization (`LightGCNConfig`)

In `gusye1234/LightGCN-PyTorch/code/utils.py` (`BPRLoss.stageOne`), each mini-batch step computes:
```python
loss, reg_loss = model.bpr_loss(users, pos, neg)
reg_loss = reg_loss * weight_decay
loss = loss + reg_loss
```
Below, `run_lightgcn_pipeline` trains `LightGCN` using the shared `SGLDataset`, `PairwiseSampler`, and `Evaluator`.

In [7]:
# ==============================================================================
# 5. Hyperparameter Configuration & End-to-End LightGCN Training / Evaluation
# ==============================================================================

@dataclass
class LightGCNConfig:
    # Dataset settings (loads ONLY `<dataset>.train` and splits 7:1:2 into train:val:test)
    data_dir: str = "./dataset/yelp2018"
    dataset: str = "yelp2018"
    split_ratios: tuple[float, float, float] = (0.7, 0.1, 0.2)
    seed: int = 2021

    # LightGCN Model Hyperparameters (matching gusye1234/LightGCN-PyTorch/code/parse.py)
    latent_dim_rec: int = 64        # Embedding dimension d (--recdim=64)
    lightGCN_n_layers: int = 3      # Number of LightGCN layers K (--layer=3)
    decay: float = 1e-4             # L2 regularization weight_decay (--decay=1e-4)
    dropout: int = 0                # Whether to use edge dropout on Graph (0 or 1)
    keep_prob: float = 0.6          # Keep probability when dropout=1 (--keepprob=0.6)
    A_split: bool = False           # Whether to split adjacency matrix into folds
    pretrain: int = 0               # Whether to use pretrained weights
    param_init: str = "normal"      # 'normal' (N(0, 0.1) in LightGCN-PyTorch) or 'xavier_uniform'

    # Optimization & Evaluation settings
    lr: float = 0.001               # Learning rate for Adam optimizer (--lr=0.001)
    batch_size: int = 2048          # Training batch size B (--bpr_batch=2048)
    epochs: int = 30                # Number of training epochs
    verbose: int = 1                # Evaluate on Val split every `verbose` epochs
    stop_cnt: int = 10              # Early stopping patience on Val Recall@20
    top_k: list[int] = field(default_factory=lambda: [20])
    test_batch_size: int = 128
    start_testing_epoch: int = 0

    def to_model_dict(self) -> dict:
        """Converts dataclass to the `config` dict expected by `LightGCN.__init__`."""
        return {
            "latent_dim_rec": self.latent_dim_rec,
            "lightGCN_n_layers": self.lightGCN_n_layers,
            "keep_prob": self.keep_prob,
            "A_split": self.A_split,
            "dropout": self.dropout,
            "pretrain": self.pretrain,
            "param_init": self.param_init,
        }


def run_lightgcn_pipeline(cfg: LightGCNConfig):
    """Runs the complete LightGCN preprocessing (7:1:2 split), training, val/test evaluation, and plotting pipeline."""
    set_seed(cfg.seed)

    print("=" * 80)
    print("LightGCN: Simplifying and Powering GCN for Recommendation (SIGIR '20)")
    print("=" * 80)
    print(
        f"[Config] dataset={cfg.dataset} | n_layers={cfg.lightGCN_n_layers} | "
        f"latent_dim={cfg.latent_dim_rec} | lr={cfg.lr} | decay={cfg.decay} | "
        f"batch_size={cfg.batch_size} | dropout={cfg.dropout}"
    )
    print(f"[Device] Using {DEVICE}")

    # 1. Load `<dataset>.train` ONLY, split 7:1:2 into train/val/test, and build graph G on 70% train
    dataset = SGLDataset(
        data_dir=cfg.data_dir,
        dataset_name=cfg.dataset,
        split_ratios=cfg.split_ratios,
        seed=cfg.seed,
        device=DEVICE,
    )
    print(
        f"[Data]   #Users: {dataset.num_users} | #Items: {dataset.num_items} | "
        f"#Total Interactions (from {cfg.dataset}.train): {dataset.total_interactions}"
    )
    print(
        f"[Split]  train:val:test = {cfg.split_ratios[0]:.0%}:{cfg.split_ratios[1]:.0%}:{cfg.split_ratios[2]:.0%} "
        f"(7:1:2) -> #Train: {dataset.num_ratings} | #Val: {dataset.num_val_ratings} | #Test: {dataset.num_test_ratings}"
    )

    data_iter = PairwiseSampler(dataset, batch_size=cfg.batch_size, shuffle=True)

    # Validation evaluator: masks 70% train items, evaluates on 10% validation items
    val_evaluator = Evaluator(
        user_train_dict=dataset.train_user_dict,
        user_test_dict=dataset.val_user_dict,
        metrics=["Precision", "Recall", "NDCG"],
        top_k=cfg.top_k,
        batch_size=cfg.test_batch_size,
    )
    # Test evaluator: masks 80% (train + val) items, evaluates on 20% held-out test items
    test_evaluator = Evaluator(
        user_train_dict=dataset.train_val_user_dict,
        user_test_dict=dataset.test_user_dict,
        metrics=["Precision", "Recall", "NDCG"],
        top_k=cfg.top_k,
        batch_size=cfg.test_batch_size,
    )

    # 2. Initialize LightGCN model & Adam optimizer (matching utils.BPRLoss in LightGCN-PyTorch)
    model = LightGCN(config=cfg.to_model_dict(), dataset=dataset).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    print("-" * 80)
    print("Val Metrics:	" + val_evaluator.metrics_info())
    print("-" * 80)

    best_epoch = 0
    best_val_result = np.zeros(3 * len(cfg.top_k), dtype=np.float64)
    best_state_dict = None
    stopping_step = 0

    history = {"epoch": [], "bpr_loss": [], "reg_loss": [], "recall": [], "ndcg": []}

    # 3. Training & Validation Loop
    for epoch in range(1, cfg.epochs + 1):
        total_loss, total_bpr_loss, total_reg_loss = 0.0, 0.0, 0.0
        num_batches = 0
        t0 = time()

        model.train()
        for bat_users, bat_pos_items, bat_neg_items in data_iter:
            bat_users = torch.from_numpy(bat_users).long().to(DEVICE)
            bat_pos_items = torch.from_numpy(bat_pos_items).long().to(DEVICE)
            bat_neg_items = torch.from_numpy(bat_neg_items).long().to(DEVICE)

            # BPRLoss.stageOne from gusye1234/LightGCN-PyTorch/code/utils.py
            bpr_l, reg_l = model.bpr_loss(bat_users, bat_pos_items, bat_neg_items)
            weighted_reg_l = reg_l * cfg.decay
            loss = bpr_l + weighted_reg_l

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_bpr_loss += bpr_l.item()
            total_reg_loss += weighted_reg_l.item()
            num_batches += 1

        avg_total = total_loss / max(1, num_batches)
        avg_bpr = total_bpr_loss / max(1, num_batches)
        avg_reg = total_reg_loss / max(1, num_batches)

        print(
            f"[iter {epoch:2d} : loss : {avg_total:.4f} = {avg_bpr:.4f} (BPR) + "
            f"{avg_reg:.4f} (Reg), time: {time() - t0:.2f}s]"
        )

        if epoch % cfg.verbose == 0 and epoch > cfg.start_testing_epoch:
            val_result, buf = val_evaluator.evaluate(model)
            print(f"epoch {epoch:2d} [Val]:	{buf}")

            history["epoch"].append(epoch)
            history["bpr_loss"].append(avg_bpr)
            history["reg_loss"].append(avg_reg)
            history["recall"].append(val_result[1])
            history["ndcg"].append(val_result[2])

            if best_val_result[1] < val_result[1]:
                best_val_result = val_result
                best_epoch = epoch
                stopping_step = 0
                best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                stopping_step += 1
                if stopping_step >= cfg.stop_cnt:
                    print(f"Early stopping triggered at epoch: {epoch}")
                    break

    # 4. Restore best validation model & evaluate on the 20% Held-Out Test Split
    if best_state_dict is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state_dict.items()})

    test_result, _ = test_evaluator.evaluate(model)
    group_recalls = test_evaluator.evaluate_long_tail_groups(
        model=model,
        train_items=dataset.train_items,
        num_items=dataset.num_items,
        num_groups=10,
        k=cfg.top_k[0],
    )

    print("=" * 80)
    print(f"Best Validation Result @ Epoch {best_epoch} (10% Val Split):")
    print("Metrics:	" + val_evaluator.metrics_info())
    print("Values: 	" + "\t".join([("%.8f" % x).ljust(12) for x in best_val_result]))
    print("-" * 80)
    print(f"Final Test Result @ Best Epoch {best_epoch} (20% Test Split):")
    print("Metrics:	" + test_evaluator.metrics_info())
    print("Values: 	" + "\t".join([("%.8f" % x).ljust(12) for x in test_result]))
    print("-" * 80)
    group_str = " | ".join([f"G{g+1}: {val:.4f}" for g, val in enumerate(group_recalls)])
    print(f"Test Long-Tail Group Recall@{cfg.top_k[0]}: {group_str}")
    print("=" * 80)

    # 5. Plot Training Curves & Test Long-Tail Group Recall
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

    axes[0].plot(history["epoch"], history["bpr_loss"], marker="o", label="BPR Loss")
    axes[0].plot(history["epoch"], history["reg_loss"], marker="s", label="L2 Reg Loss")
    axes[0].set_title("LightGCN Training Losses per Epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Batch-Averaged Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history["epoch"], history["recall"], marker="o", color="tab:green", label=f"Val Recall@{cfg.top_k[0]}")
    axes[1].plot(history["epoch"], history["ndcg"], marker="^", color="tab:purple", label=f"Val NDCG@{cfg.top_k[0]}")
    axes[1].set_title(f"LightGCN Top-{cfg.top_k[0]} Validation Performance")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Metric Value")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    axes[2].bar(np.arange(1, 11), group_recalls, color="tab:orange", alpha=0.85)
    axes[2].set_title(f"LightGCN Test Long-Tail Group Recall@{cfg.top_k[0]} (Fig. 4)")
    axes[2].set_xlabel("GroupID (1 = Long-Tail, 10 = Popular Head)")
    axes[2].set_ylabel("Group Recall Contribution")
    axes[2].set_xticks(np.arange(1, 11))
    axes[2].grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

    return model, history, group_recalls


# Execute the LightGCN pipeline
config = LightGCNConfig(
    latent_dim_rec=64,
    lightGCN_n_layers=3,
    decay=1e-4,
    lr=0.001,
    epochs=3,
)
trained_lgn_model, lgn_history, lgn_long_tail_recalls = run_lightgcn_pipeline(config)
